# Persistence - SSD

SSDs are not just “faster disks.” They have their own constraints:
- storage is organized into banks, blocks, and pages
- reads and writes happen at page granularity
- erases happen at block granularity
- a page cannot simply be overwritten in place
- flash cells wear out after enough writes

That last point is the key design challenge: reads are easy, writes are awkward, erases are expensive.

---

## 🟢 SSD operation costs: why reads, writes, and erases behave differently
The timing slide is trying to show an important hierarchy:
- SRAM / DRAM: nanoseconds
- flash: microseconds
- disk: milliseconds

So flash is much slower than RAM, but much faster than disk. That is why SSDs feel dramatically faster than HDDs, especially for random access.

Within flash itself:
- read page is relatively fast
- program/write page is slower
- erase block is much slower

So an SSD write is not symmetric with an SSD read. A read can directly fetch a page. A write may require:
- finding erased space
- programming a new page there
- later reclaiming the old version
- eventually erasing blocks during cleanup

This is why SSD internals are heavily optimized around writes.

---

## 🟢 Why flash cannot overwrite in place
In magnetic disks, you can overwrite a sector in place.
In flash, you generally cannot rewrite an already-programmed page directly. The page must first belong to a block that has been erased.
- erase is block-wide, but read/write is page-wide

That mismatch is what creates complexity.

---

## 🟢 Cell density: SLC, MLC, TLC
The density slide is really about a three-way tradeoff:
- capacity
- speed
- endurance/reliability

### SLC - Single-Level Cell
- stores 1 bit per cell
- simplest to distinguish electrically
- fastest
- most durable
- most expensive

### MLC - Multi-Level Cel
- stores 2 bits per cell using more charge levels
- denser and cheaper per bit
- slower than SLC
- lower endurance

### TLC - Triple-Level Cell
- stores 3 bits per cell
- even denser and cheaper
- slower still
- more sensitive to wear and write/erase stress

Higher density lowers cost per bit, but usually hurts latency and endurance.

---

## 🟢 Why SSDs need an FTL at all
The host system wants to see a normal block device:
- “read block 100”
- “write block 2001”

But the flash hardware underneath has awkward constraints:
- page-sized reads/writes
- block-sized erases
- no in-place overwrites
- wear limits

### The Flash Translation Layer (FTL) exists to hide that mismatch.
Its job is to translate:
- logical block/page addresses used by the OS
into
- physical flash locations and operations

So the SSD controller and FTL make flash behave like a normal block device, even though internally it is not one.

### The FTL is responsible for:
- logical-to-physical mapping
- deciding where new writes go
- garbage collection
- wear leveling
- keeping mapping metadata persistent
- exploiting multiple chips in parallel

That is why SSD performance depends heavily on controller design, not just raw flash chips.

---

## 🟢Direct-mapped FTL vs Log-structured FTL
### A. Direct-mapped organization
The simplest idea is:
- logical page x lives at physical page x

Reads are easy:
- just go to the corresponding physical page

Writes are terrible:
- to modify one page, you may need to
- read the whole block
- erase the block
- rewrite the changed page plus the unchanged old pages

That causes severe write amplification and poor endurance.
A tiny logical write can trigger a large amount of physical flash traffic.

So direct mapping is conceptually simple but operationally bad.

### B. Log-structured organization
The better idea is:
- never overwrite old data in place
- write the new version to the next free page
- update the mapping table so the logical page now points to the new physical page

So if logical page 2001 used to map to physical page 3, the later update might map it to physical page 4.

Now writes become cheap in the short term:
- just append the new version somewhere free

Reads require consulting the mapping table:
- logical page x might be at physical page y

This is much better for performance because it avoids constant erase-before-write at the moment of every overwrite.

---

## 🟢 Valid, invalid, erased pages: what those states mean

The garbage-collection slides use states like:
- valid
- invalid/dead
- erased/free

These are crucial.

### Valid
Contains the current version of some logical page.

### Invalid/dead
Contains an old version that has been superseded by a newer write somewhere else.

### Erased/free

Can accept a new program/write operation.

This means SSD blocks naturally evolve into mixed states:
- some pages still live
- some pages obsolete
- some pages free

That is why background cleanup is unavoidable.

---

## 🟢 Garbage collection: why SSDs slow down under writes
Because log-structured writes keep appending new versions, the SSD accumulates old invalid pages.
Eventually, it must reclaim space.

Garbage collection process
- find a block with some invalid pages
- copy its remaining live pages elsewhere
- erase the old block
- return it to the free pool

This is expensive because:
- copying live pages costs extra reads/writes
- block erase is slow
- it creates internal SSD traffic the host did not ask for

That is the hidden cost of making writes look easy.

The important insight:
- log-structured writes defer the erase cost, but do not eliminate it.
Garbage collection pays the bill later.

---

## 🟢 Write amplification: the core SSD penalty metric

Write amplification formula: total bytes written internally to flash / total bytes written by the client
This is one of the most important SSD concepts.

### Example
If the host writes 4 KB, but the SSD ends up internally moving/writing 16 KB because of:
- metadata updates
- garbage collection
- live-page migration
- block merges

then write amplification is:
```
16 / 4 = 4
```
High write amplification is bad because it means:
- lower performance
- more flash wear
- more background work

So FTL design tries to keep write amplification low.

---

## 🟢 Wear leveling: why SSDs must spread writes around

Flash pages wear out after enough erase/program cycles.
If one region gets rewritten constantly while another region is barely touched, the hot region dies early.

---